In [ ]:
import pandas as pd
import numpy as np
import time
from geopy.geocoders import Nominatim
from opencage.geocoder import OpenCageGeocode

# =====================================================================
# DATA PIPELINE: PREPROCESSING, FILTERING, & GEOCODING
# =====================================================================

print("1. Memuat dan Membersihkan Data Mentah...")
# Ganti nama file ini jika data mentahmu bernama lain
df = pd.read_csv('Data pelanggan subzona 215.csv')
df['JALAN'] = df['JALAN'].astype(str).str.strip()
df['THBL'] = df['THBL'].astype(str)

print("2. Melakukan Agregasi Data per Jalan dan Bulan...")
df_agg = df.groupby(['THBL', 'JALAN']).agg(
    TOTAL_PAKAI=('PAKAI', 'sum'),
    TOTAL_RP=('RP_PAKAI', 'sum'),
    JUMLAH_PELANGGAN=('NO_PLG', 'count')
).reset_index()

print("3. Memfilter Anomali Harga (Rp 800 - Rp 12.999 per m³)...")
# Hitung Harga per m3 (mencegah infinity jika TOTAL_PAKAI = 0)
df_agg['HARGA_PER_M3'] = np.where(
    df_agg['TOTAL_PAKAI'] > 0,
    df_agg['TOTAL_RP'] / df_agg['TOTAL_PAKAI'],
    0
)

jml_awal = len(df_agg)
# Menerapkan Filter Batas Harga sesuai permintaan
df_clean = df_agg[(df_agg['HARGA_PER_M3'] > 800) & (df_agg['HARGA_PER_M3'] <= 12999)].copy()
jml_akhir = len(df_clean)
print(f"   -> Total data sebelum filter : {jml_awal} baris")
print(f"   -> Total data setelah filter : {jml_akhir} baris")
print(f"   -> Berhasil menghapus {jml_awal - jml_akhir} baris data anomali.")

print("\n4. Mempersiapkan Mesin Geocoding...")
# Mengambil jalan unik dari data yang sudah BERSIH
unique_streets = pd.DataFrame(df_clean['JALAN'].unique(), columns=['JALAN'])

# Inisialisasi API Geocoding
geolocator_nominatim = Nominatim(user_agent="pdam_clustering_sby")
API_KEY_OPENCAGE = "6e06f3a7c172486f8f91a66d2e28c196"
geocoder_opencage = OpenCageGeocode(API_KEY_OPENCAGE)

# Fungsi cerdas (Pipeline Geocoding Bertingkat)
def get_smart_coordinates(jalan):
    lat, lon = np.nan, np.nan
    
    # Tahap 1: Coba Nominatim (Gratis)
    try:
        query_nom = f"Jalan {jalan}, Surabaya, Jawa Timur"
        loc = geolocator_nominatim.geocode(query_nom, timeout=5)
        time.sleep(0.5) # Jeda agar tidak diblokir
        if loc:
            lat, lon = loc.latitude, loc.longitude
    except:
        pass

    # Tahap 2: Coba OpenCage jika Nominatim Gagal
    if pd.isna(lat):
        try:
            query_oc = f"{jalan}, Surabaya, Jawa Timur, Indonesia"
            res = geocoder_opencage.geocode(query_oc)
            time.sleep(0.5)
            if res and len(res) > 0:
                lat = res[0]['geometry']['lat']
                lon = res[0]['geometry']['lng']
        except:
            pass
            
    # Tahap 3: Koreksi Kilat 
    # (Jika titik jatuh di pusat generic Surabaya [-7.24917] atau tetap gagal)
    if pd.isna(lat) or round(lat, 5) == -7.24917:
        j_up = str(jalan).upper()
        if 'ITS' in j_up:
            return pd.Series([-7.281111, 112.794611])
        elif 'GEBANG' in j_up or 'RODAH' in j_up:
            return pd.Series([-7.279833, 112.796322])
        elif 'KEPUTIH' in j_up or 'ARIF' in j_up:
            return pd.Series([-7.288531, 112.798155])
        elif 'MARINA' in j_up:
            return pd.Series([-7.281512, 112.802871])
            
    return pd.Series([lat, lon])

print(f"5. Memproses pencarian koordinat untuk {len(unique_streets)} jalan unik (Mohon tunggu)...")
unique_streets[['Latitude', 'Longitude']] = unique_streets['JALAN'].apply(get_smart_coordinates)

print("6. Menggabungkan dan Menyimpan File Akhir...")
# Gabungkan koordinat dengan data historis bulanan yang sudah bersih
df_final = pd.merge(df_clean, unique_streets, on='JALAN', how='left')

# Simpan ke CSV final
nama_file = "Data_Siap_Clustering_Final3.csv"
df_final.to_csv(nama_file, index=False)
print(f"✅ BERHASIL! File '{nama_file}' sudah bersih, akurat, dan siap digunakan di Dashboard.")